<a href="https://colab.research.google.com/github/liviooneil/2026_Intro_Python/blob/main/live/part-I/1.6-geospatial-vector-data-practice-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.6-geospatial-vector-data-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Practice Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [2]:
%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Your solution here
exo_df = pd.DataFrame({"name":["A","B"], "ion":["7.4","8.5"], "lat":["46.9","47.4"],
                       })
exo_df

,name,ion,lat
0,A,7.4,46.9
1,B,8.5,47.4


## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [7]:
# Your solution here

from pathlib import Path
exo_df = gpd.GeoDataFrame(
    exo_df,
    geometry=gpd.points_from_xy(exo_df["ion"], exo_df["lat"]),
    crs="EPSG:4326",   # WGS84 longitude/latitude in degrees
)

Path("_files").mkdir(exist_ok=True)
exo_df.to_file("_files/exo_df.geojson", driver="GeoJSON")
exo_df.to_file("_files/exo_df.gpkg", driver="GPKG")

reread = gpd.read_file("_files/exo_df.gpkg")
print("reread shape:", reread.shape, "| crs preserved:", reread.crs.to_epsg())

reread shape: (2, 4) | crs preserved: 4326


## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [8]:
# Your solution here
exo_df = exo_df.to_crs(2056)   # reproject to metric Swiss coordinates

print("before:", exo_df.crs.to_epsg(), "->", "after:", exo_df.crs.to_epsg())
print("Exo WGS84 (deg):", round(exo_df.geometry.x.iloc[0], 3), round(exo_df.geometry.y.iloc[0], 3))
print("Exo LV95 (m):  ", round(exo_df.geometry.x.iloc[0], 1), round(exo_df.geometry.y.iloc[0], 1))

before: 2056 -> after: 2056
Exo WGS84 (deg): 2597056.203 1194321.876
Exo LV95 (m):   2597056.2 1194321.9


## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [15]:
# Your solution here

lv95 = exo_df.to_crs(2056)
A = lv95.loc[lv95["name"] == "A", "geometry"].iloc[0]
B = lv95.loc[lv95["name"] == "B", "geometry"].iloc[0]
print(round(A.distance(B) / 1000, 1), "km")

100.2 km


## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [18]:
# Your solution here
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")

# which stations fall within a hazard zone?
in_hazard = gpd.sjoin(exo_df, zone, predicate="within", how="inner")
print("stations in a hazard zone:", in_hazard["name"].tolist())

print(zone.contains(exo_df.geometry.iloc[0]).tolist())

NameError: name 'ion' is not defined

## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
# Your solution here

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here